# Day 32 — Practice Questions
### Getting Data Into Shape

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A is pen-and-paper first.** The numbers are chosen to be easy. Write your answer,
*then* run the cell to check.

Three questions to keep asking yourself:

- Does this model measure **distance**? If yes, it needs scaling.
- Did I **split before** anything learned from the data?
- Does this new feature actually **move the score**?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("ready")

---
# Part A — Work it out

Eight questions. Most need no computer.

**The three formulas:**

| Scaler | Formula |
|---|---|
| standard | (x − mean) ÷ sd |
| min-max | (x − min) ÷ range |
| robust | (x − median) ÷ IQR |

### A1 · Standard scaling by hand

Five numbers. Work out the mean and the sd, then scale them.

*(Use the population sd — divide by n, not n−1. That is what scikit-learn does.)*

In [ ]:
x = np.array([10., 20., 30., 40., 50.])

print("mean :", x.mean())
print("sd   :", x.std())
print("scaled:", StandardScaler().fit_transform(x.reshape(-1, 1)).ravel().round(3))

### A2 · Min-max by hand

Same five numbers.

In [ ]:
print("min:", x.min(), "  max:", x.max(), "  range:", x.max() - x.min())
print("scaled:", MinMaxScaler().fit_transform(x.reshape(-1, 1)).ravel().round(3))

### A3 · Now add one wild value

Same five numbers, plus a sixth: **500**. What happens to the first five under min-max?

In [ ]:
x2 = np.array([10., 20., 30., 40., 50., 500.])
scaled = MinMaxScaler().fit_transform(x2.reshape(-1, 1)).ravel()

print("scaled:", scaled.round(4))
print()
print("the five original values are now spread over just",
      round(scaled[:5].max() - scaled[:5].min(), 4))

### A4 · Robust scaling, same wild value

Try the same six numbers with `RobustScaler`, which uses the **median** and the **IQR**
(the width of the middle half of the data) instead.

In [ ]:
q1, med, q3 = np.percentile(x2, [25, 50, 75])
print("median:", med, "   IQR:", q3 - q1)

rb = RobustScaler().fit_transform(x2.reshape(-1, 1)).ravel()
print("scaled:", rb.round(3))
print()
print("the five original values are now spread over",
      round(rb[:5].max() - rb[:5].min(), 3))

### A5 · Which of these needs scaling?

For each model, say yes or no — and why.

| | Model |
|---|---|
| a | k-nearest neighbours |
| b | Random forest |
| c | k-means clustering |
| d | Decision tree |

### A6 · Which of these could a model work out on its own?

For each new column, say whether a linear model could have found it by itself.

| | The new column |
|---|---|
| a | `maths + science` |
| b | `bill ÷ items` |
| c | `2 × age` |
| d | `is_weekend` (from a date) |

### A7 · What order should these go in?

Put these four steps in the correct order:

- scale the columns
- hide 30% of the rows
- build a `price_per_item` column
- train the model

### A8 · How many columns is too many?

You have **200 rows**. After one-hot encoding and some feature engineering you have **180
columns**.

What will happen, and what should you do?

In [ ]:
print("rows    :", 200)
print("columns :", 180)
print("rows per column:", round(200 / 180, 2))

---
# Part B — Find the mistake

Four short programs. Each one runs cleanly and gives a misleading answer.

### B1 · The score is terrible and nobody knows why

In [ ]:
rng = np.random.default_rng(601)
n = 400
area  = rng.normal(900, 250, n).clip(300, 2000)        # square feet
baths = rng.integers(1, 4, n).astype(float)            # 1 to 3
rent  = 4000 + 22*area + 3000*baths + rng.normal(0, 2500, n)

X = np.column_stack([area, baths])
X_train, X_hidden, y_train, y_hidden = train_test_split(X, rent, test_size=.3, random_state=0)

model = KNeighborsRegressor(7).fit(X_train, y_train)   # <-- look here
print("score:", round(model.score(X_hidden, y_hidden), 4))

### B2 · Somebody scaled a random forest

In [ ]:
X_train, X_hidden, y_train, y_hidden = train_test_split(X, rent, test_size=.3, random_state=0)
scaler = StandardScaler().fit(X_train)

plain  = RandomForestRegressor(n_estimators=80, random_state=0).fit(X_train, y_train)
scaled = RandomForestRegressor(n_estimators=80, random_state=0).fit(
             scaler.transform(X_train), y_train)

print("forest without scaling:", round(plain.score(X_hidden, y_hidden), 4))
print("forest with scaling   :", round(scaled.score(scaler.transform(X_hidden), y_hidden), 4))

### B3 · The scaler saw rows it should not have

In [ ]:
rng = np.random.default_rng(602)
n = 400
Xb = rng.normal(50, 12, (n, 2))
yb = Xb[:, 0]*0.6 + Xb[:, 1]*0.3 + rng.normal(0, 4, n)

Xb_scaled = StandardScaler().fit_transform(Xb)        # <-- look here

a, b, c, d = train_test_split(Xb_scaled, yb, test_size=.3, random_state=0)
print("score:", round(KNeighborsRegressor(7).fit(a, c).score(b, d), 4))

### B4 · The same mistake, four hundred times more expensive

In [ ]:
rng = np.random.default_rng(603)
X_junk = rng.normal(0, 1, (120, 2500))
y_junk = rng.normal(0, 1, 120)            # nothing on earth predicts this

# pick the ten most promising columns, using ALL the rows
best = SelectKBest(f_regression, k=10).fit(X_junk, y_junk).get_support()

print("score:", round(cross_val_score(Ridge(), X_junk[:, best], y_junk, cv=5).mean(), 3))
print("(remember: this data is pure random noise. The honest answer is 0.)")

---
# Part C — Write the code

Five tasks on one dataset.

In [ ]:
# ---------------------------------------------------------------
# 800 food deliveries. Run this once - C1 to C5 all use `orders`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3232)
n = 800

items    = rng.integers(1, 15, n).astype(float)
per_item = rng.normal(190, 70, n).clip(50, None)
bill     = (items * per_item).round(0)
distance = rng.uniform(0.5, 12, n).round(2)                 # km
ordered  = pd.to_datetime("2025-03-01") + pd.to_timedelta(rng.integers(0, 24*90, n), "h")

is_dinner = ((ordered.hour >= 19) & (ordered.hour <= 22)).astype(float)
minutes   = (14 + 2.4*distance + 9*is_dinner + 0.02*items + rng.normal(0, 3.5, n)).round(1)

orders = pd.DataFrame({"bill": bill, "items": items, "distance_km": distance,
                       "ordered_at": ordered, "minutes": minutes})
orders.head()

### C1 · Does this model need scaling?

Fit a `KNeighborsRegressor(7)` to predict `minutes` from `bill`, `items` and
`distance_km` — **once without scaling and once with**. Report both scores on hidden rows.

Remember: split first.

In [ ]:
# your code here

### C2 · Try all three scalers

Run the same kNN model with `StandardScaler`, `MinMaxScaler` and `RobustScaler`. Report
all three scores and say which you would choose here.

In [ ]:
# your code here

### C3 · Build two new features

Add these to the table, then measure whether each one helped:

1. `per_item` — the bill divided by the number of items
2. `is_dinner` — 1 if the order was placed between 7pm and 10pm, otherwise 0

Use scaled kNN, and add them **one at a time**.

In [ ]:
# your code here

### C4 · Is a raw timestamp any use?

Add the timestamp as a plain number (seconds since the 1st of March) and check whether it
helps. Compare it against `is_dinner`.

In [ ]:
# your code here

### C5 · Put the whole thing together

Build your best model: the useful features only, scaled properly, split properly. Report:

- the score on hidden rows
- the average error, in minutes
- how much of that error is irreducible *(look at how the data was built)*

In [ ]:
# your code here

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Build five features and keep only the good ones

Below is a raw table of 1,000 online shoppers. A model using the raw columns scores
poorly, because the things that actually drive spending are **not** any single column.

**Your tasks:**

1. Engineer **five** new features. Before measuring each one, write a sentence saying what
   real-world idea it captures.
2. Add them **one at a time** and record the score after each.
3. Keep only the ones that earned their place, and report your final score.

Hints, in increasing order of giving it away: think about **per-visit** amounts; think
about **how long ago** something happened; think about what `first_order` really encodes.

In [ ]:
rng = np.random.default_rng(777)
n = 1000

first_order = pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, n), "D")
days_known  = (pd.Timestamp("2025-09-01") - first_order).days.values.astype(float)
n_visits    = rng.poisson(days_known / 95 + 2).astype(float)
total_spent = n_visits * rng.normal(540, 160, n).clip(100, None)
n_returns   = rng.binomial(n_visits.astype(int), 0.11).astype(float)
support_msgs = rng.poisson(n_visits * 0.18 + 0.5).astype(float)

# the truth: value depends on SPEND RATE and BASKET SIZE, hurt by the RETURN RATE
basket    = np.divide(total_spent, n_visits, out=np.full(n, 540.0), where=n_visits > 0)
rate      = total_spent / np.maximum(days_known, 30)
ret_rate  = np.divide(n_returns, n_visits, out=np.zeros(n), where=n_visits > 0)

value = (900 + 380*rate + 2.1*basket - 1400*ret_rate - 40*support_msgs
         + rng.normal(0, 180, n)).round(0)

shoppers = pd.DataFrame({"first_order": first_order, "n_visits": n_visits,
                         "total_spent": total_spent.round(0), "n_returns": n_returns,
                         "support_msgs": support_msgs, "value": value})
shoppers.head()

In [ ]:
# the starting point: the raw number columns, scaled
raw_cols = ["n_visits", "total_spent", "n_returns", "support_msgs"]

def score_shoppers(column_list, data):
    a, b, c, d = train_test_split(data[column_list], data["value"],
                                  test_size=0.3, random_state=0)
    sc = StandardScaler().fit(a)
    return Ridge().fit(sc.transform(a), c).score(sc.transform(b), d)

print("starting score:", round(score_shoppers(raw_cols, shoppers), 4))

In [ ]:
# your five features here

### D2 · A model that scores 0.97. Prove it is worth nothing.

A colleague hands you the model below. It predicts electricity bills with a score of
**0.97** on rows it has never seen. The scaling is correct, the split is correct, the code
has no bugs.

**Your tasks:**

1. Run it and confirm the score.
2. Work out why it is useless. It is **not** a scaling mistake and **not** a split mistake.
3. Prove it: rebuild from the honest columns and report the real score.
4. Write two or three sentences you could send your colleague.

*Hint: look at each column and ask when its value actually becomes known.*

In [ ]:
rng = np.random.default_rng(2468)
n = 600

rooms     = rng.integers(1, 7, n).astype(float)
area_sqft = (rooms * rng.normal(230, 50, n)).clip(250, None).round(0)
residents = np.clip(np.round(rooms * 0.8 + rng.normal(0, 0.9, n)), 1, 9)
has_ac    = rng.binomial(1, 0.45, n)

units = (55 + 16*rooms + 0.045*area_sqft + 14*residents + 90*has_ac
         + rng.normal(0, 18, n)).clip(20, None)        # units of electricity used
bill  = (units * 7.4 + 120 + rng.normal(0, 25, n)).round(0)

# the meter reading, recorded by the meter man at the end of the month
units_consumed = units.round(1)

homes = pd.DataFrame({"rooms": rooms, "area_sqft": area_sqft, "residents": residents,
                      "has_ac": has_ac, "units_consumed": units_consumed, "bill": bill})
homes.head()

In [ ]:
cols = ["rooms", "area_sqft", "residents", "has_ac", "units_consumed"]

a, b, c, d = train_test_split(homes[cols], homes["bill"], test_size=0.3, random_state=0)
sc = StandardScaler().fit(a)
m = Ridge().fit(sc.transform(a), c)

print(f"score on hidden rows : {m.score(sc.transform(b), d):.4f}")
print(f"average error        : ₹{mean_absolute_error(d, m.predict(sc.transform(b))):,.2f}")

In [ ]:
# your investigation here

---
## Before you hand it in

- For every model: does it **measure distance**? If yes, did you scale?
- Did you **split before** the scaler learned anything?
- Did you use `.fit()` on the training rows and `.transform()` on both?
- For every new feature: did you add it **one at a time** and check the score?
- Did you **delete** the features that did not help?
- For every column: would its value be **known at the moment you need the prediction**?

Three numbers worth carrying away:

- Scaling took a kNN model from **−0.13 to 0.68**. The same scaling changed a random forest
  by **exactly nothing**.
- A leaked **scaler** cost **0.0007**. A leaked **column picker** cost **0.62**. The same
  mistake, nearly a thousand times the damage.
- Engineered features took a model from **0.21 to 0.99** — with no new data collected.
- And two of those features were worth nothing at all. You only knew because you added
  them **one at a time**.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*